# Лабораторна робота №2 - Частина 1
## Наука про дані: підготовчий етап

**Мета:** Завантаження, очищення та аналіз VHI-індексу по областях України за допомогою NOAA та pandas.

In [1]:
import os
import urllib.request
from datetime import datetime
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

### 1. Завантаження даних
Процедура завантажує файли для 27 адміністративних одиниць (уникаючи provinceID=0). Імена зберігаються з датою. Вбудовано перевірку на наявність вже завантажених файлів.

In [2]:
def download_vhi_data(directory="vhi_data"):
    if not os.path.exists(directory):
        os.makedirs(directory)
        
    now = datetime.now().strftime("%Y%m%d_%H%M%S")
    downloaded_files = []
    
    for province_id in range(1, 28):
        # Перевірка на наявність вже завантаженого файлу
        existing_files = [f for f in os.listdir(directory) if f.startswith(f"vhi_id_{province_id}_")]
        if existing_files:
            print(f"Файл для області {province_id} вже існує. Пропускаємо.")
            downloaded_files.append(os.path.join(directory, existing_files[0]))
            continue
            
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        
        try:
            req = urllib.request.urlopen(url)
            text = req.read().decode('utf-8')
            
            filename = f"vhi_id_{province_id}_{now}.csv"
            filepath = os.path.join(directory, filename)
            
            with open(filepath, 'w') as f:
                f.write(text)
            
            print(f"Завантажено: {filename}")
            downloaded_files.append(filepath)
            
        except Exception as e:
            print(f"Помилка завантаження для області {province_id}: {e}")
            
    return downloaded_files

# Запускаємо завантаження
files = download_vhi_data()

Файл для області 1 вже існує. Пропускаємо.
Файл для області 2 вже існує. Пропускаємо.
Файл для області 3 вже існує. Пропускаємо.
Файл для області 4 вже існує. Пропускаємо.
Файл для області 5 вже існує. Пропускаємо.
Файл для області 6 вже існує. Пропускаємо.
Файл для області 7 вже існує. Пропускаємо.
Файл для області 8 вже існує. Пропускаємо.
Файл для області 9 вже існує. Пропускаємо.
Файл для області 10 вже існує. Пропускаємо.
Файл для області 11 вже існує. Пропускаємо.
Файл для області 12 вже існує. Пропускаємо.
Файл для області 13 вже існує. Пропускаємо.
Файл для області 14 вже існує. Пропускаємо.
Файл для області 15 вже існує. Пропускаємо.
Файл для області 16 вже існує. Пропускаємо.
Файл для області 17 вже існує. Пропускаємо.
Файл для області 18 вже існує. Пропускаємо.
Файл для області 19 вже існує. Пропускаємо.
Файл для області 20 вже існує. Пропускаємо.
Файл для області 21 вже існує. Пропускаємо.
Файл для області 22 вже існує. Пропускаємо.
Файл для області 23 вже існує. Пропускаєм

### 2. Зчитування, очищення даних (Data Cleaning) та зміна індексів
Прибираємо HTML-теги, заповнюємо/видаляємо порожні значення та міняємо індекси областей на порядок згідно з українською абеткою.

In [3]:
import os
import pandas as pd

def prepare_dataframe(directory="vhi_data"):
    noaa_to_ua_index = {
        1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
        11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16,
        20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5
    }
    
    province_names = {
        1: 'Вінницька', 2: 'Волинська', 3: 'Дніпропетровська', 4: 'Донецька',
        5: 'Житомирська', 6: 'Закарпатська', 7: 'Запорізька', 8: 'Івано-Франківська',
        9: 'Київська', 10: 'Кіровоградська', 11: 'Луганська', 12: 'Львівська',
        13: 'Миколаївська', 14: 'Одеська', 15: 'Полтавська', 16: 'Рівненська',
        17: 'Сумська', 18: 'Тернопільська', 19: 'Харківська', 20: 'Херсонська',
        21: 'Хмельницька', 22: 'Черкаська', 23: 'Чернівецька', 24: 'Чернігівська',
        25: 'Республіка Крим', 26: 'м. Київ', 27: 'м. Севастополь'
    }

    all_dfs = []
    
    for file in os.listdir(directory):
        if not file.endswith(".csv"):
            continue
            
        filepath = os.path.join(directory, file)
        old_id = int(file.split('_')[2])
        new_id = noaa_to_ua_index.get(old_id)
        
        if new_id is None:
            continue
            
        # Зчитування
        df_temp = pd.read_csv(filepath, header=1, names=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty'], skipinitialspace=True)
        
        # Видаляємо порожній стовпець
        df_temp.drop(columns=['empty'], inplace=True, errors='ignore')
        
        # ОЧИЩЕННЯ: Видаляємо всі HTML-теги (наприклад <tt><pre>) з колонки Year за допомогою регулярного виразу
        df_temp['Year'] = df_temp['Year'].astype(str).str.replace(r'<[^>]+>', '', regex=True)
        # Прибираємо можливі пробіли
        df_temp['Year'] = df_temp['Year'].str.strip()
        
        # Залишаємо тільки ті рядки, де Year складається з цифр (ігноруємо сміття)
        df_temp = df_temp[df_temp['Year'].str.isnumeric()]
        df_temp.dropna(how='any', inplace=True)
        
        # ТЕПЕР БЕЗПЕЧНО: Перетворення типів
        df_temp = df_temp.astype(float)
        df_temp['Year'] = df_temp['Year'].astype(int)
        df_temp['Week'] = df_temp['Week'].astype(int)
        
        # Заміна -1 (відсутні дані) на NaN і видалення
        df_temp['VHI'].replace(-1.0, pd.NA, inplace=True)
        df_temp.dropna(subset=['VHI'], inplace=True)
        
        # Додавання індексів
        df_temp['Province_ID'] = new_id
        df_temp['Province_Name'] = province_names[new_id]
        
        all_dfs.append(df_temp)
        
    df_final = pd.concat(all_dfs, ignore_index=True)
    return df_final

# Створюємо головний DataFrame
df = prepare_dataframe()
print("Дані очищено! Приклад:")
display(df.sample(5))

Дані очищено! Перші 5 записів:


,Year,Week,SMN,SMT,VCI,TCI,VHI,Province_ID,Province_Name
52779,2007,52,0.058,263.28,62.17,20.40,41.29,4,Донецька
52635,2005,12,0.115,277.21,59.11,44.52,51.82,4,Донецька
49618,1990,11,0.194,286.11,94.48,1.55,48.01,3,Дніпропетровська
45156,1990,21,0.434,291.93,63.18,59.41,61.29,23,Чернівецька
16040,1989,25,0.314,299.32,39.57,66.31,52.94,14,Одеська


### 3. Процедури формування вибірок
Нижче реалізовані окремі функції для фільтрації та пошуку необхідних показників.

In [4]:
def get_vhi_for_province_and_year(df, province_id, year):
    result = df[(df['Province_ID'] == province_id) & (df['Year'] == year)][['Week', 'VHI']]
    return result

def get_vhi_for_provinces_and_years(df, province_ids, year_start, year_end):
    result = df[(df['Province_ID'].isin(province_ids)) & (df['Year'] >= year_start) & (df['Year'] <= year_end)]
    return result[['Year', 'Week', 'Province_Name', 'VHI']]

def get_vhi_stats(df, province_ids, years):
    filtered = df[(df['Province_ID'].isin(province_ids)) & (df['Year'].isin(years))]
    stats = {
        'Min VHI': filtered['VHI'].min(),
        'Max VHI': filtered['VHI'].max(),
        'Mean VHI': filtered['VHI'].mean(),
        'Median VHI': filtered['VHI'].median()
    }
    return pd.DataFrame([stats])

print("1. VHI для 1-ї області (Вінницька) за 2010 рік (перші 5 тижнів):")
display(get_vhi_for_province_and_year(df, province_id=1, year=2010).head())

print("\n2. VHI для Київської(9) та Львівської(12) областей за 2015-2016 роки (перші 5 записів):")
display(get_vhi_for_provinces_and_years(df, province_ids=[9, 12], year_start=2015, year_end=2016).head())

print("\n3. Статистика для Одеської(14) області за 2020 рік:")
display(get_vhi_stats(df, province_ids=[14], years=[2020]))

1. VHI для 1-ї області (Вінницька) за 2010 рік (перші 5 тижнів):


,Week,VHI
34996,1,52.04
34997,2,51.05
34998,3,52.37
34999,4,52.69
35000,5,53.16



2. VHI для Київської(9) та Львівської(12) областей за 2015-2016 роки (перші 5 записів):


,Year,Week,Province_Name,VHI
3952,2015,1,Київська,46.27
3953,2015,2,Київська,47.79
3954,2015,3,Київська,48.26
3955,2015,4,Київська,47.77
3956,2015,5,Київська,46.46



3. Статистика для Одеської(14) області за 2020 рік:


,Min VHI,Max VHI,Mean VHI,Median VHI
0,22.77,46.71,37.631154,40.43
